# V4 Dominant Subspace PCG Sanity

- Same dataset and kernel settings as v1_mat32_eps_topq_eigenspace_compare
- Build dominant subspace preconditioner on GPU and run PCG
- Print timing, residual, and accuracy metrics

In [10]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here.parent,
    _here.parent.parent,
    _here.parent.parent.parent,
    Path("D:/NU/ML"),
]
for p in _candidates:
    pkg_dir = p / "efgp_eigenpro_py"
    if pkg_dir.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

from efgp_eigenpro_py.benchmark import (
    make_dataset,
    make_test_set,
    true_func_2d,
    compute_rmse,
)
from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py.gpu.v3_eigenspace import EigenspaceConfig, estimate_top_eigenspace_v3
from efgp_eigenpro_py.kernels import make_matern
from efgp_eigenpro_py.gpu.backends import BackendConfig, build_gpu_backend_bundle
from efgp_eigenpro_py.gpu.contexts import GPUOperatorContext, ensure_gpu_data_context
from efgp_eigenpro_py.gpu.iterative_solvers import pcg_solve_gpu
from efgp_eigenpro_py.gpu.v1_ops import (
    apply_A_v1,
    gpu_precompute_v1,
    predict_v1,
    solve_beta_plain_cg_v1,
)
from efgp_eigenpro_py.gpu.v2_preconditioner import (
    apply_preconditioner_dominant_subspace,
    apply_preconditioner_v2,
    build_dominant_subspace_preconditioner,
    build_gpu_preconditioner_data,
)

np.set_printoptions(precision=6, suppress=True)
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])

cwd: d:\NU\ML\efgp_eigenpro_py\gpu\sanity_check
sys.path[0]: D:\NU\ML


In [11]:
import gc

try:
    import cupy as cp

    cp.cuda.Stream.null.synchronize()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()
except Exception:
    pass

gc.collect()
print("GPU memory pool cleared (if available).")

GPU memory pool cleared (if available).


In [12]:
REG_LAMBDA = 0.1
N_TRAIN = 100_000
DIM = 2
LENGTHSCALE = 0.1
NU = 1.5
EPS = 5e-6

Q_LIST_DOM = [15,30,45,60,75,80]
Q_LIST_BASE = [16, 32, 48,64,80]
S_OVERSAMPLE = 4
KMAX = 1
KEEP_FACTOR = 5.0

EIG_METHOD = "subspace_iter"
EIG_N_ITER = 3
EIG_BLOCK_SIZE = None
EIG_OVERSAMPLE = 8

N_TEST = 20_000
NOISE = 0.02
SEED_TRAIN = 20230
SEED_TEST = 2027

SOLVE_TOL = 1e-6
GPU_TOL = SOLVE_TOL
GPU_MAXITER = 3000
GPU_CHUNK_SIZE = None
GPU_DEBUG_FINITE_CHECKS = False
GPU_NUFFT = "auto"
L2_SCALED = True

print("EPS:", EPS)
print("Q_LIST_DOM:", Q_LIST_DOM)
print("Q_LIST_BASE:", Q_LIST_BASE)
print("S_OVERSAMPLE:", S_OVERSAMPLE)
print("KMAX:", KMAX)
print("KEEP_FACTOR:", KEEP_FACTOR)
print("EIG_METHOD:", EIG_METHOD)
print("EIG_N_ITER:", EIG_N_ITER)
print("EIG_OVERSAMPLE:", EIG_OVERSAMPLE)
print("EIG_BLOCK_SIZE:", EIG_BLOCK_SIZE)

EPS: 5e-06
Q_LIST_DOM: [15, 30, 45, 60, 75, 80]
Q_LIST_BASE: [16, 32, 48, 64, 80]
S_OVERSAMPLE: 4
KMAX: 1
KEEP_FACTOR: 5.0
EIG_METHOD: subspace_iter
EIG_N_ITER: 3
EIG_OVERSAMPLE: 8
EIG_BLOCK_SIZE: None


In [13]:
print("Building dataset ...")
x_train, y_train = make_dataset(DIM, N_TRAIN, true_func_2d, noise=NOISE, seed=SEED_TRAIN)
x_test, y_test = make_test_set(DIM, N_TEST, true_func_2d, seed=SEED_TEST)

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test:", x_test.shape, x_test.dtype)

kernel = make_matern(lengthscale=LENGTHSCALE, nu=NU, dim=DIM, variance=1.0)
print(kernel)

solver = EFGPSolver(
    kernel=kernel,
    reg_lambda=REG_LAMBDA,
    eps=EPS,
    nufft_tol=1e-10,
    l2scaled=L2_SCALED,
)

Building dataset ...
x_train: (100000, 2) float64
y_train: (100000,) float64
x_test: (20000, 2) float64
KernelSpec(fam='matern', dim=2, lengthscale=0.1, variance=1.0, nu=1.5)


In [14]:
def _device_to_numpy(xp, arr):
    if hasattr(xp, "asnumpy"):
        return np.asarray(xp.asnumpy(arr))
    if hasattr(arr, "get"):
        return np.asarray(arr.get())
    return np.asarray(arr)


def _mse_mae(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    diff = a - b
    mse = float(np.mean(np.abs(diff) ** 2))
    mae = float(np.mean(np.abs(diff)))
    return mse, mae

In [15]:
backend = build_gpu_backend_bundle(BackendConfig(nufft=GPU_NUFFT))
data_ctx = ensure_gpu_data_context(backend, x_train, y_train, state=None)
data_ctx.meta["debug_finite_checks"] = bool(GPU_DEBUG_FINITE_CHECKS)
op_ctx = GPUOperatorContext()

t0 = time.perf_counter()
data_ctx = gpu_precompute_v1(
    backend,
    solver.kernel,
    solver.eps,
    solver.nufft_tol,
    data_ctx,
    op_ctx,
    l2scaled=solver.l2scaled,
    chunk_size=GPU_CHUNK_SIZE,
)
t1 = time.perf_counter()
precompute_time = float(t1 - t0)

state_cpu = solver.precompute(x_train, y_train)
print("precompute_time:", precompute_time)

precompute_time: 0.4570800999645144


In [16]:
def _apply_A_block(v_block):
    xp = backend.xp
    v_block = xp.asarray(v_block)
    if v_block.ndim == 1:
        out = xp.empty_like(v_block)
        apply_A_v1(backend, data_ctx, v_block, float(REG_LAMBDA), op_ctx, out=out)
        return out
    if v_block.ndim != 2:
        raise ValueError("v_block must be 1D or 2D.")

    w_flat = data_ctx.weights_gpu_flat
    gf = data_ctx.gf_gpu
    if w_flat is None or gf is None:
        raise RuntimeError("gpu_precompute_v1 must run before apply_A_block.")

    mtot = int(data_ctx.meta.get("mtot", 0))
    dim = int(data_ctx.meta.get("dim", 0))
    if mtot <= 0 or dim <= 0:
        raise RuntimeError("Invalid mtot/dim for apply_A_block.")

    n = int(v_block.shape[0])
    k = int(v_block.shape[1])
    if n != int(mtot) ** int(dim):
        raise ValueError("v_block leading dimension mismatch.")

    dtype = xp.result_type(v_block.dtype, gf.dtype)
    v_block = v_block.astype(dtype, copy=False)
    w_flat = xp.asarray(w_flat, dtype=dtype)
    gf = xp.asarray(gf, dtype=dtype)

    shape = (mtot,) * dim
    v_nd = v_block.reshape(shape + (k,))
    w_nd = w_flat.reshape(shape + (1,))
    wv = w_nd * v_nd

    pad_shape = tuple(int(s) for s in gf.shape) + (k,)
    pad = xp.zeros(pad_shape, dtype=dtype)
    pad_slicer = tuple(slice(0, mtot) for _ in range(dim)) + (slice(None),)
    pad[pad_slicer] = wv

    axes = tuple(range(dim))
    af = backend.fft.fftn(pad, axes=axes)
    vfft = backend.fft.ifftn(af * gf[..., None], axes=axes)
    out_slicer = tuple(slice(mtot - 1, 2 * mtot - 1) for _ in range(dim)) + (slice(None),)
    t = vfft[out_slicer].reshape(n, k)

    out = w_flat.reshape(-1, 1) * t
    out += float(REG_LAMBDA) * v_block
    return out

In [17]:
rows = []

for q in Q_LIST_DOM:
    q = int(q)
    if q <= 0:
        continue

    t_pb0 = time.perf_counter()
    precond_data, precond_diag = build_dominant_subspace_preconditioner(
        backend=backend,
        apply_A_block=_apply_A_block,
        size=int(data_ctx.rhs_gpu.size),
        sigma2=float(REG_LAMBDA),
        q=q,
        s=S_OVERSAMPLE,
        kmax=KMAX,
        keep_factor=KEEP_FACTOR,
    )
    t_pb1 = time.perf_counter()

    kept_rank = int(precond_diag.get("kept_rank", 0))
    res_fro = np.nan
    res_rel = np.nan
    if kept_rank > 0:
        xp = backend.xp
        U = precond_data.U_gpu
        inv_shift = precond_data.inv_shift_gpu
        theta = 1.0 / (inv_shift + 1.0)
        AU = _apply_A_block(U)
        resid = AU - U * theta.reshape(1, -1)
        res_fro = float(xp.linalg.norm(resid))
        au_norm = float(xp.linalg.norm(AU))
        res_rel = res_fro / max(au_norm, 1e-30)

    def _matvec(v, out):
        apply_A_v1(backend, data_ctx, v, float(REG_LAMBDA), op_ctx, out=out)

    def _precond(v, out):
        apply_preconditioner_dominant_subspace(
            backend,
            precond_data,
            v,
            op_ctx=op_ctx,
            out=out,
        )

    t_s0 = time.perf_counter()
    beta_gpu, it, relres, stats = pcg_solve_gpu(
        backend,
        _matvec,
        _precond,
        data_ctx.rhs_gpu,
        op_ctx,
        GPU_TOL,
        GPU_MAXITER,
        return_stats=True,
    )
    t_s1 = time.perf_counter()

    t_pg0 = time.perf_counter()
    yhat_gpu = predict_v1(backend, data_ctx, x_test, beta_gpu)
    yhat_gpu_host = _device_to_numpy(backend.xp, yhat_gpu)
    t_pg1 = time.perf_counter()

    beta_cpu = _device_to_numpy(backend.xp, beta_gpu)
    yhat_cpu = solver.predict(x_test, beta_cpu, state_cpu)
    rmse = compute_rmse(yhat_cpu, y_test)
    yhat_mse, yhat_mae = _mse_mae(yhat_gpu_host, yhat_cpu)

    budget_proxy = int(precond_diag.get("p", 0)) * (int(precond_diag.get("kmax", KMAX)) + 1)

    rows.append(
        {
            "q": q,
            "p": int(precond_diag.get("p", 0)),
            "kept_rank": kept_rank,
            "build_rank_used": int(kept_rank),
            "build_budget_proxy": int(budget_proxy),
            "time_precompute": precompute_time,
            "time_precond_build": float(t_pb1 - t_pb0),
            "time_solve": float(t_s1 - t_s0),
            "time_predict": float(t_pg1 - t_pg0),
            "time_total": float(
                precompute_time + (t_pb1 - t_pb0) + (t_s1 - t_s0) + (t_pg1 - t_pg0)
            ),
            "cg_iters": int(it),
            "cg_relres": float(relres),
            "n_matvec": int(stats["n_matvec"]),
            "t_matvec_avg": float(stats["t_matvec_avg"]),
            "t_matvec_total": float(stats["t_matvec_total"]),
            "n_precond": int(stats["n_precond"]),
            "t_precond_avg": float(stats["t_precond_avg"]),
            "t_precond_total": float(stats["t_precond_total"]),
            "rmse_test": float(rmse),
            "yhat_mse": float(yhat_mse),
            "yhat_mae": float(yhat_mae),
            "subspace_residual_fro": float(res_fro),
            "subspace_residual_rel": float(res_rel),
        }
    )

df = pd.DataFrame(rows)
print("rows:", len(df))
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(df)

rows: 6


,q,p,kept_rank,build_rank_used,build_budget_proxy,time_precompute,time_precond_build,time_solve,time_predict,time_total,cg_iters,cg_relres,n_matvec,t_matvec_avg,t_matvec_total,n_precond,t_precond_avg,t_precond_total,rmse_test,yhat_mse,yhat_mae,subspace_residual_fro,subspace_residual_rel
0,15,19,19,19,38,0.45708,0.398085,3.940784,0.009058,4.805007,505,9.810548e-07,506,0.004914,2.486493,505,0.001380,0.696919,0.002601,0.0,0.0,2984.092707,0.240228
1,30,34,34,34,68,0.45708,0.455086,3.634704,0.008925,4.555795,450,9.671891e-07,451,0.004978,2.245035,450,0.001541,0.693409,0.002601,0.0,0.0,2803.445052,0.207850
2,45,49,49,49,98,0.45708,1.094282,6.608968,0.009859,8.170189,368,9.586061e-07,369,0.005120,1.889292,368,0.010658,3.922078,0.002601,0.0,0.0,2199.744449,0.156943
3,60,64,64,64,128,0.45708,2.148138,23.054028,0.012324,25.671570,307,9.857597e-07,308,0.023397,7.206246,307,0.042033,12.904030,0.002601,0.0,0.0,1590.537976,0.111615
4,75,79,79,79,158,0.45708,15.389409,11.218783,0.013119,27.078391,259,9.528131e-07,260,0.023371,6.076507,259,0.010203,2.642698,0.002601,0.0,0.0,1274.307953,0.088772
5,80,84,84,84,168,0.45708,11.921547,13.260158,0.052924,25.691710,248,9.647732e-07,249,0.033224,8.272767,248,0.010786,2.674863,0.002601,0.0,0.0,1162.030841,0.080858


In [18]:
import gc

xp = backend.xp
for name in ("precond_data", "U", "inv_shift", "theta", "AU", "resid"):
    if name in globals():
        del globals()[name]
if hasattr(xp, "cuda"):
    try:
        xp.cuda.Stream.null.synchronize()
    except Exception:
        pass
gc.collect()
if hasattr(xp, "get_default_memory_pool"):
    xp.get_default_memory_pool().free_all_blocks()
if hasattr(xp, "get_default_pinned_memory_pool"):
    xp.get_default_pinned_memory_pool().free_all_blocks()

rows_baseline = []


def _resolve_block_size(q_req):
    block = EIG_BLOCK_SIZE
    if block is None:
        block = max(q_req + EIG_OVERSAMPLE, q_req + 1)
    return int(block)


for q in Q_LIST_BASE:
    q = int(q)
    if q <= 0:
        continue

    q_req = q + 1
    block_size = _resolve_block_size(q_req)
    eig_cfg = EigenspaceConfig(
        q_max=q_req,
        block_size=block_size,
        n_iter=EIG_N_ITER,
        method=EIG_METHOD,
    )

    t_e0 = time.perf_counter()
    vals_gpu, vecs_gpu, eig_diag = estimate_top_eigenspace_v3(
        backend=backend,
        apply_A_block_gpu=_apply_A_block,
        size=int(data_ctx.rhs_gpu.size),
        cfg=eig_cfg,
    )
    t_e1 = time.perf_counter()

    if vals_gpu.size > q:
        mu = float(vals_gpu[q])
    else:
        mu = float(vals_gpu[-1])
    scale_gpu = 1.0 - (mu / vals_gpu[:q])

    t_pb0 = time.perf_counter()
    precond_v2 = build_gpu_preconditioner_data(backend, vecs_gpu[:, :q], scale_gpu)
    t_pb1 = time.perf_counter()

    xp = backend.xp
    U_gpu = precond_v2.U_gpu
    AU = _apply_A_block(U_gpu)
    resid = AU - U_gpu * xp.asarray(vals_gpu[:q]).reshape(1, -1)
    res_fro = float(xp.linalg.norm(resid))
    au_norm = float(xp.linalg.norm(AU))
    res_rel = res_fro / max(au_norm, 1e-30)

    n_iter = int(eig_diag.get("n_iter", EIG_N_ITER))
    block_used = int(eig_diag.get("block_size", block_size))
    budget_proxy = block_used * (n_iter + 2)

    def _matvec(v, out):
        apply_A_v1(backend, data_ctx, v, float(REG_LAMBDA), op_ctx, out=out)

    def _precond(v, out):
        apply_preconditioner_v2(backend, precond_v2, v, op_ctx=op_ctx, out=out)

    t_s0 = time.perf_counter()
    beta_gpu, it, relres, stats = pcg_solve_gpu(
        backend,
        _matvec,
        _precond,
        data_ctx.rhs_gpu,
        op_ctx,
        GPU_TOL,
        GPU_MAXITER,
        return_stats=True,
    )
    t_s1 = time.perf_counter()

    t_pg0 = time.perf_counter()
    yhat_gpu = predict_v1(backend, data_ctx, x_test, beta_gpu)
    yhat_gpu_host = _device_to_numpy(backend.xp, yhat_gpu)
    t_pg1 = time.perf_counter()

    beta_cpu = _device_to_numpy(backend.xp, beta_gpu)
    yhat_cpu = solver.predict(x_test, beta_cpu, state_cpu)
    rmse = compute_rmse(yhat_cpu, y_test)
    yhat_mse, yhat_mae = _mse_mae(yhat_gpu_host, yhat_cpu)

    rows_baseline.append(
        {
            "method": "v3_gpu_eigenspace",
            "q": q,
            "p": int(q),
            "kept_rank": int(q),
            "build_rank_used": int(q),
            "build_budget_proxy": int(budget_proxy),
            "time_precompute": precompute_time,
            "time_eigenspace": float(t_e1 - t_e0),
            "time_precond_build": float(t_pb1 - t_pb0),
            "time_solve": float(t_s1 - t_s0),
            "time_predict": float(t_pg1 - t_pg0),
            "time_total": float(
                precompute_time + (t_e1 - t_e0) + (t_pb1 - t_pb0) + (t_s1 - t_s0) + (t_pg1 - t_pg0)
            ),
            "cg_iters": int(it),
            "cg_relres": float(relres),
            "n_matvec": int(stats["n_matvec"]),
            "t_matvec_avg": float(stats["t_matvec_avg"]),
            "t_matvec_total": float(stats["t_matvec_total"]),
            "n_precond": int(stats["n_precond"]),
            "t_precond_avg": float(stats["t_precond_avg"]),
            "t_precond_total": float(stats["t_precond_total"]),
            "rmse_test": float(rmse),
            "yhat_mse": float(yhat_mse),
            "yhat_mae": float(yhat_mae),
            "eig_residual_fro": float(res_fro),
            "eig_residual_rel": float(res_rel),
        }
    )


t_v10 = time.perf_counter()
beta_gpu, it, relres, stats = solve_beta_plain_cg_v1(
    backend,
    data_ctx,
    float(REG_LAMBDA),
    op_ctx,
    GPU_TOL,
    GPU_MAXITER,
    return_stats=True,
)
t_v11 = time.perf_counter()

t_vp0 = time.perf_counter()
yhat_gpu = predict_v1(backend, data_ctx, x_test, beta_gpu)
yhat_gpu_host = _device_to_numpy(backend.xp, yhat_gpu)
t_vp1 = time.perf_counter()

beta_cpu = _device_to_numpy(backend.xp, beta_gpu)
yhat_cpu = solver.predict(x_test, beta_cpu, state_cpu)
rmse = compute_rmse(yhat_cpu, y_test)
yhat_mse, yhat_mae = _mse_mae(yhat_gpu_host, yhat_cpu)

v1_row = {
    "method": "v1_plain_cg",
    "q": 0,
    "p": 0,
    "kept_rank": 0,
    "build_rank_used": 0,
    "build_budget_proxy": 0,
    "time_precompute": precompute_time,
    "time_eigenspace": 0.0,
    "time_precond_build": 0.0,
    "time_solve": float(t_v11 - t_v10),
    "time_predict": float(t_vp1 - t_vp0),
    "time_total": float(precompute_time + (t_v11 - t_v10) + (t_vp1 - t_vp0)),
    "cg_iters": int(it),
    "cg_relres": float(relres),
    "n_matvec": int(stats["n_matvec"]),
    "t_matvec_avg": float(stats["t_matvec_avg"]),
    "t_matvec_total": float(stats["t_matvec_total"]),
    "n_precond": 0,
    "t_precond_avg": 0.0,
    "t_precond_total": 0.0,
    "rmse_test": float(rmse),
    "yhat_mse": float(yhat_mse),
    "yhat_mae": float(yhat_mae),
    "eig_residual_fro": np.nan,
    "eig_residual_rel": np.nan,
}

df_dom = df.copy()
df_dom["method"] = "dominant_subspace"
df_dom["time_eigenspace"] = df_dom["time_precond_build"]
df_dom["eig_residual_fro"] = df_dom["subspace_residual_fro"]
df_dom["eig_residual_rel"] = df_dom["subspace_residual_rel"]

df_base = pd.DataFrame(rows_baseline)
df_v1 = pd.DataFrame([v1_row])
compare_method_order = {
    "v1_plain_cg": 0,
    "v3_gpu_eigenspace": 1,
    "dominant_subspace": 2,
}
df_compare = pd.concat([df_v1, df_base, df_dom], ignore_index=True, sort=False)
df_compare["section_order"] = df_compare["method"].map(compare_method_order)
df_compare = df_compare.sort_values(by=["section_order", "q", "p"]).drop(
    columns=["section_order"]
)
df_compare = df_compare.reset_index(drop=True)
print("compare rows:", len(df_compare))
with pd.option_context("display.max_columns", None, "display.width", 200):
    display(df_compare)

compare rows: 12


,method,q,p,kept_rank,build_rank_used,build_budget_proxy,time_precompute,time_eigenspace,time_precond_build,time_solve,time_predict,time_total,cg_iters,cg_relres,n_matvec,t_matvec_avg,t_matvec_total,n_precond,t_precond_avg,t_precond_total,rmse_test,yhat_mse,yhat_mae,eig_residual_fro,eig_residual_rel,subspace_residual_fro,subspace_residual_rel
0,v1_plain_cg,0,0,0,0,0,0.45708,0.000000,0.000000,2.422274,0.009986,2.889340,403,9.297732e-07,404,0.004979,2.011408,0,0.000000,0.000000,0.002601,0.0,0.0,NaN,NaN,NaN,NaN
1,v3_gpu_eigenspace,16,16,16,16,125,0.45708,4.237510,0.000181,6.710684,0.010655,11.416111,297,9.278551e-07,298,0.011865,3.535801,297,0.007047,2.093017,0.002601,0.0,0.0,412.006584,0.031029,NaN,NaN
2,v3_gpu_eigenspace,32,32,32,32,205,0.45708,1.220076,0.000163,1.601770,0.008948,3.288038,209,9.879828e-07,210,0.004819,1.011944,209,0.001398,0.292115,0.002602,0.0,0.0,260.579671,0.018418,NaN,NaN
3,v3_gpu_eigenspace,48,48,48,48,285,0.45708,2.754628,0.000237,1.529041,0.012268,4.753255,169,9.904419e-07,170,0.005288,0.898897,169,0.001958,0.330862,0.002602,0.0,0.0,210.301576,0.014651,NaN,NaN
4,v3_gpu_eigenspace,64,64,64,64,365,0.45708,8.006791,0.000154,1.092784,0.009689,9.566499,135,9.790942e-07,136,0.004817,0.655078,135,0.001847,0.249331,0.002602,0.0,0.0,153.054564,0.010611,NaN,NaN
5,v3_gpu_eigenspace,80,80,80,80,445,0.45708,12.909362,0.000154,0.943322,0.009006,14.318925,114,9.815053e-07,115,0.004750,0.546301,114,0.002052,0.233894,0.002602,0.0,0.0,109.157705,0.007553,NaN,NaN
6,dominant_subspace,15,19,19,19,38,0.45708,0.398085,0.398085,3.940784,0.009058,4.805007,505,9.810548e-07,506,0.004914,2.486493,505,0.001380,0.696919,0.002601,0.0,0.0,2984.092707,0.240228,2984.092707,0.240228
7,dominant_subspace,30,34,34,34,68,0.45708,0.455086,0.455086,3.634704,0.008925,4.555795,450,9.671891e-07,451,0.004978,2.245035,450,0.001541,0.693409,0.002601,0.0,0.0,2803.445052,0.207850,2803.445052,0.207850
8,dominant_subspace,45,49,49,49,98,0.45708,1.094282,1.094282,6.608968,0.009859,8.170189,368,9.586061e-07,369,0.005120,1.889292,368,0.010658,3.922078,0.002601,0.0,0.0,2199.744449,0.156943,2199.744449,0.156943
9,dominant_subspace,60,64,64,64,128,0.45708,2.148138,2.148138,23.054028,0.012324,25.671570,307,9.857597e-07,308,0.023397,7.206246,307,0.042033,12.904030,0.002601,0.0,0.0,1590.537976,0.111615,1590.537976,0.111615
